# NSMC 감성 분류: 처음부터 다시 구현

상위 폴더의 `nsmc_kcbert_classification.ipynb`를 이해한 뒤, 참고 코드를 보지 않고 빈 노트북에서 전체 파이프라인을 다시 작성한 버전입니다. 구현하며 찾은 버그와 배운 점은 [README](../README.md#처음부터-다시-구현하기-from_scratch)에 정리했습니다.

> 실행 결과는 전체 데이터로 다시 학습한 뒤 추가할 예정입니다.

In [ ]:
# 0. 환경 설정
import torch          # PyTorch
import transformers   # Hugging Face 사전학습 모델 라이브러리 (PyTorch 위에서 동작)

In [ ]:
# 1. 설정값
from dataclasses import dataclass

@dataclass  # 데코레이터: 이 클래스에 dataclass 기능을 붙임 (__init__, __repr__ 자동 생성)
class Config:
    model_name: str = "beomi/kcbert-base"  # 사전학습 모델
    max_length: int = 128                  # 문장당 최대 토큰 수
    batch_size: int = 32                   # 한 번에 학습하는 문장 수
    learning_rate: float = 5e-5
    epochs: int = 1
    warmup_ratio: float = 0.1              # 전체 스텝 중 학습률을 천천히 올리는 구간 비율
    val_ratio: float = 0.1                 # 학습 데이터에서 떼어 낼 검증 데이터 비율
    train_size: int | None = None          # 빠른 실험용 학습 데이터 크기 (None = 전체)
    seed: int = 42
    save_dir: str = "nsmc-kcbert"

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU가 있으면 cuda, 없으면 cpu
use_amp = device.type == "cuda"  # GPU일 때만 AMP(혼합 정밀도) 사용

In [ ]:
# 2. 랜덤 시드 고정: cfg.seed로 random, numpy, torch의 난수를 모두 고정해 결과를 재현
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)           # CPU
    torch.cuda.manual_seed_all(seed)  # 모든 GPU

set_seed(cfg.seed)

In [ ]:
# 3. 데이터 불러오기 (NSMC 원본: 탭 구분, id / document / label)
import csv
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/"
# 리뷰 속 따옴표(")가 칸 묶음 기호로 해석되지 않도록 QUOTE_NONE
train_df = pd.read_csv(BASE_URL + "ratings_train.txt", sep="\t", quoting=csv.QUOTE_NONE)
test_df = pd.read_csv(BASE_URL + "ratings_test.txt", sep="\t", quoting=csv.QUOTE_NONE)

# 빈 리뷰(NaN) 제거 후 인덱스 재정렬
train_df = train_df.dropna(subset=["document"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["document"]).reset_index(drop=True)

In [ ]:
# 4. 학습 / 검증 분리
# ① (선택) 빠른 실험용: 학습 데이터 크기 줄이기
if cfg.train_size is not None:
    train_df = train_df.sample(n=cfg.train_size, random_state=cfg.seed).reset_index(drop=True)

# ② (항상) 학습 데이터에서 검증 데이터 10% 떼어 내기
val_df = train_df.sample(frac=cfg.val_ratio, random_state=cfg.seed)  # 원래 인덱스를 유지한 채 무작위 추출
train_df = train_df.drop(val_df.index).reset_index(drop=True)         # 뽑힌 줄을 학습에서 제외
val_df = val_df.reset_index(drop=True)                                # 다 쓴 뒤에 인덱스 정리

print("train:", len(train_df), "/ val:", len(val_df))

In [ ]:
# 5. 토크나이저 불러오기
from transformers import AutoTokenizer

# cfg.model_name에 맞는 토크나이저(KcBERT WordPiece)를 불러옴
# 문장을 넣어 호출하면 input_ids, token_type_ids, attention_mask를 돌려줌
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

In [ ]:
# 6. 데이터셋과 데이터로더
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):  # train / val / test 모두 이 클래스 하나로 처리
    def __init__(self, df, tokenizer, max_length):  # df: 전처리된 데이터, tokenizer: 위에서 불러온 토크나이저
        self.encodings = tokenizer(
            df["document"].tolist(),  # 토크나이저 입력용 Python 리스트로 변환
            truncation=True,          # max_length보다 길면 자르기
            max_length=max_length,
            padding="max_length",     # 짧으면 [PAD]로 채워 길이 맞추기
        )
        self.labels = df["label"].tolist()  # 정답 레이블도 리스트로 저장

    def __len__(self):  # len(dataset) → 샘플 수
        return len(self.labels)

    def __getitem__(self, idx):  # dataset[idx] → idx번째 샘플 (DataLoader가 배치 크기만큼 호출해 묶음)
        # 키 세 개를 돌면서 idx번째 리뷰 것만 꺼내 텐서로 바꿔 item에 저장
        item = {key: torch.tensor(values[idx]) for key, values in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])  # idx번째 정답 레이블 추가
        return item

# 학습 / 검증 / 테스트 데이터셋 생성
train_dataset = NSMCDataset(train_df, tokenizer, cfg.max_length)
val_dataset = NSMCDataset(val_df, tokenizer, cfg.max_length)
test_dataset = NSMCDataset(test_df, tokenizer, cfg.max_length)

# DataLoader(데이터셋, 배치 크기, 섞기 여부, num_workers = 데이터를 미리 준비할 프로세스 수)
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=2)

In [ ]:
# 7. 모델 초기화
from transformers import AutoModelForSequenceClassification

# 사전학습된 KcBERT 위에 분류 헤드(클래스 2개: 부정/긍정)를 얹은 모델 생성
# Auto 클래스가 모델 이름을 보고 BertForSequenceClassification을 골라 줌
model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=2)
model.to(device)  # 모든 가중치를 GPU 메모리로 이동

In [ ]:
# 8. 옵티마이저와 학습률 스케줄러
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=cfg.learning_rate)  # 모델의 모든 가중치를 학습 대상으로
total_steps = len(train_loader) * cfg.epochs        # 전체 스텝 = 에폭당 배치 수 × 에폭 수
warmup_steps = int(total_steps * cfg.warmup_ratio)  # 학습률을 0에서 천천히 올리는 구간
# warmup 동안 학습률을 올린 뒤, 끝까지 선형으로 줄임
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

In [ ]:
# 9. 평가 함수
import torch.nn.functional as F  # 손실 함수, 활성화 함수 등 함수 형태의 신경망 연산

@torch.no_grad()  # 평가 때는 기울기를 계산하지 않아 메모리와 시간 절약
def evaluate(model, loader):
    model.eval()  # 평가 모드: 드롭아웃 끄기
    total_loss, correct, total = 0.0, 0, 0  # 손실 합계, 맞힌 개수, 전체 개수
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}  # 배치의 텐서를 모두 GPU로
        labels = batch.pop("labels")  # 정답 텐서를 꺼내고 batch에서는 제거 (모델 입력만 남김)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):  # AMP: 가능한 계산은 float16으로
            logits = model(**batch).logits          # 리뷰마다 [부정 점수, 긍정 점수]
            loss = F.cross_entropy(logits, labels)  # 배치의 평균 손실
        total_loss += loss.item() * labels.size(0)  # 평균 × 개수 = 이 배치의 손실 합계
        correct += (logits.argmax(dim=-1) == labels).sum().item()  # 맞힌 개수 누적
        total += labels.size(0)  # 평가한 리뷰 수 누적
    model.train()  # 학습 모드로 되돌리기
    return total_loss / total, correct / total  # (평균 손실, 정확도)

In [ ]:
# 10. 학습 루프
from tqdm.auto import tqdm

# AMP(Automatic Mixed Precision, 자동 혼합 정밀도)용 스케일러:
# float16에서 아주 작은 기울기가 0이 되지 않도록 손실을 키웠다가 업데이트 전에 되돌림
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
model.train()

for epoch in range(cfg.epochs):  # 에폭 수만큼 반복
    progress = tqdm(train_loader, desc=f"epoch {epoch + 1}/{cfg.epochs}")  # 진행 막대
    for step, batch in enumerate(progress):
        batch = {key: value.to(device) for key, value in batch.items()}
        labels = batch.pop("labels")

        optimizer.zero_grad()                       # 1. 이전 스텝의 기울기 초기화 (배치마다)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits          # 2. 순전파 (예측값 계산)
            loss = F.cross_entropy(logits, labels)  # 3. 손실 계산
        scaler.scale(loss).backward()               # 4. 역전파 (기울기 계산)
        scaler.step(optimizer)                      # 5. 가중치 업데이트
        scaler.update()                             #    다음 스텝의 손실 배율 조정
        scheduler.step()                            #    학습률 조정

        if step % 50 == 0:  # 50 스텝마다 진행 막대에 현재 손실과 학습률 표시
            progress.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    # 에폭이 끝날 때마다 검증 데이터로 평가
    val_loss, val_acc = evaluate(model, val_loader)
    print(f"[epoch {epoch + 1}] val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

In [ ]:
# 11. 최종 테스트 평가와 모델 저장
# 학습이 끝난 뒤 test로 한 번만 평가
# loss: 정답에 얼마나 확신을 갖고 맞혔는지 (낮을수록 좋음) / acc: 얼마나 많이 맞혔는지
test_loss, test_acc = evaluate(model, test_loader)
print(f"test_loss={test_loss:.4f}  test_acc={test_acc:.4f}")  # 소수점 아래 4자리까지 출력

model.save_pretrained(cfg.save_dir)      # 학습된 모델을 cfg.save_dir에 저장
tokenizer.save_pretrained(cfg.save_dir)  # 토크나이저도 함께 저장